<a href="https://colab.research.google.com/github/Geethima-Rajapaksha/ai-retail-analyst/blob/main/Analyst_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
!pip install plotly anthropic -q

In [17]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')
print("/ All libraries loaded!")

/ All libraries loaded!


In [21]:
df = pd.read_csv('retail_sales_2023_2024-2 (1).csv')
df['date'] = pd.to_datetime(df['date'])

print(f"✅ Data loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"📅 Date range: {df['date'].min().date()} → {df['date'].max().date()}")
print(f"💰 Total Revenue: ${df['revenue'].sum():,.2f}")
df.head()

✅ Data loaded: 5,017 rows × 13 columns
📅 Date range: 2023-01-01 → 2024-12-31
💰 Total Revenue: $1,258,788.60


,order_id,date,product,category,units_sold,unit_price,discount_pct,revenue,cost,profit,region,salesperson,channel
0,ORD-1000,2023-01-01,Planner Pro,Office Supplies,1,34.0,0,34.0,10,24.0,North,Alice Chen,B2B
1,ORD-1001,2023-01-01,Wireless Mouse,Electronics,1,39.0,0,39.0,12,27.0,South,Bob Martinez,In-Store
2,ORD-1002,2023-01-01,USB-C Hub,Electronics,2,53.1,10,106.2,36,70.2,North,Alice Chen,In-Store
3,ORD-1003,2023-01-02,Desk Organizer,Furniture,1,40.5,10,40.5,14,26.5,East,Henry Brown,In-Store
4,ORD-1004,2023-01-02,Cable Management Kit,Accessories,4,29.0,0,116.0,32,84.0,South,Grace Lee,Online


In [22]:
total_revenue   = df['revenue'].sum()
total_profit    = df['profit'].sum()
total_orders    = df['order_id'].nunique()
total_units     = df['units_sold'].sum()
profit_margin   = (total_profit / total_revenue) * 100
avg_order_value = total_revenue / total_orders

print("=" * 45)
print("        📊 BUSINESS KPI DASHBOARD")
print("=" * 45)
print(f"  💰 Total Revenue     : ${total_revenue:>12,.2f}")
print(f"  📈 Total Profit      : ${total_profit:>12,.2f}")
print(f"  🧾 Total Orders      : {total_orders:>12,}")
print(f"  📦 Units Sold        : {total_units:>12,}")
print(f"  💹 Profit Margin     : {profit_margin:>11.1f}%")
print(f"  🛒 Avg Order Value   : ${avg_order_value:>12.2f}")
print("=" * 45)

# --- Monthly Revenue Trend ---
monthly = df.groupby(df['date'].dt.to_period('M')).agg(
    revenue=('revenue','sum'),
    orders=('order_id','count'),
    profit=('profit','sum')
).reset_index()
monthly['date'] = monthly['date'].astype(str)

# --- Month over Month Growth ---
monthly['mom_growth'] = monthly['revenue'].pct_change() * 100

# --- Top Products ---
top_products = df.groupby('product').agg(
    revenue=('revenue','sum'),
    units=('units_sold','sum'),
    profit=('profit','sum')
).sort_values('revenue', ascending=False).reset_index()

# --- Category Breakdown ---
category_kpi = df.groupby('category').agg(
    revenue=('revenue','sum'),
    profit=('profit','sum'),
    orders=('order_id','count')
).reset_index()
category_kpi['margin'] = (category_kpi['profit'] / category_kpi['revenue'] * 100).round(1)

# --- Region Performance ---
region_kpi = df.groupby('region')['revenue'].sum().sort_values(ascending=False).reset_index()

print("\n🏆 TOP 5 PRODUCTS BY REVENUE:")
print(top_products[['product','revenue','units','profit']].head())

print("\n📂 CATEGORY BREAKDOWN:")
print(category_kpi[['category','revenue','margin']].to_string(index=False))

print("\n🗺️ REVENUE BY REGION:")
print(region_kpi.to_string(index=False))

        📊 BUSINESS KPI DASHBOARD
  💰 Total Revenue     : $1,258,788.60
  📈 Total Profit      : $  618,838.60
  🧾 Total Orders      :        5,017
  📦 Units Sold        :        8,993
  💹 Profit Margin     :        49.2%
  🛒 Avg Order Value   : $      250.90

🏆 TOP 5 PRODUCTS BY REVENUE:
                   product    revenue  units     profit
0            Laptop Pro 15  408925.20    333  149185.20
1               4K Monitor  150371.10    289   60781.10
2            Standing Desk  130513.25    182   61353.25
3  Noise-Cancel Headphones  105472.25    370   57372.25
4          Ergonomic Chair   92176.50    197   48836.50

📂 CATEGORY BREAKDOWN:
       category   revenue  margin
    Accessories  37650.95    69.1
    Electronics 829961.50    45.0
      Furniture 310356.70    53.6
Office Supplies  80819.45    65.3

🗺️ REVENUE BY REGION:
 region   revenue
  North 331430.40
  South 325975.65
   East 307321.50
Central 157885.75
   West 136175.30


In [23]:
from plotly.subplots import make_subplots

# --- Chart 1: Monthly Revenue Trend ---
fig1 = px.line(monthly, x='date', y='revenue',
               title='📈 Monthly Revenue Trend (2023-2024)',
               labels={'revenue': 'Revenue ($)', 'date': 'Month'},
               color_discrete_sequence=['#2563EB'])
fig1.update_traces(line_width=3, mode='lines+markers')
fig1.update_layout(template='plotly_white', height=400)
fig1.show()

# --- Chart 2: Top 10 Products by Revenue ---
fig2 = px.bar(top_products.head(10),
               x='revenue', y='product',
               orientation='h',
               title='🏆 Top 10 Products by Revenue',
               labels={'revenue': 'Revenue ($)', 'product': ''},
               color='revenue',
               color_continuous_scale='Blues')
fig2.update_layout(template='plotly_white', height=450)
fig2.show()

# --- Chart 3: Revenue by Category (Pie) ---
fig3 = px.pie(category_kpi, values='revenue', names='category',
              title='📂 Revenue Share by Category',
              color_discrete_sequence=px.colors.qualitative.Set2)
fig3.update_traces(textposition='inside', textinfo='percent+label')
fig3.update_layout(height=400)
fig3.show()

# --- Chart 4: Region Performance ---
fig4 = px.bar(region_kpi, x='region', y='revenue',
              title='🗺️ Revenue by Region',
              color='revenue',
              color_continuous_scale='Teal',
              labels={'revenue': 'Revenue ($)', 'region': 'Region'})
fig4.update_layout(template='plotly_white', height=400)
fig4.show()

# --- Chart 5: Monthly Profit vs Revenue ---
fig5 = go.Figure()
fig5.add_trace(go.Bar(x=monthly['date'], y=monthly['revenue'],
                       name='Revenue', marker_color='#2563EB'))
fig5.add_trace(go.Bar(x=monthly['date'], y=monthly['profit'],
                       name='Profit', marker_color='#16A34A'))
fig5.update_layout(title='💹 Monthly Revenue vs Profit',
                    barmode='group', template='plotly_white',
                    height=400,
                    xaxis_tickangle=-45)
fig5.show()

# --- Chart 6: Sales Channel Breakdown ---
channel_kpi = df.groupby('channel')['revenue'].sum().reset_index()
fig6 = px.pie(channel_kpi, values='revenue', names='channel',
              title='🛒 Revenue by Sales Channel',
              hole=0.4,
              color_discrete_sequence=px.colors.qualitative.Pastel)
fig6.update_layout(height=400)
fig6.show()

print("✅ All 6 charts generated!")


✅ All 6 charts generated!


In [24]:
def get_mock_insights():
    return f"""
📊 KEY INSIGHTS:
1. Laptop Pro 15 drives 32% of total revenue ($408K)
   — heavily dependent on one product.
2. Strong seasonality detected — Nov/Dec sales are
   60% higher than Jan/Feb.
3. Online channel leads with 40% of all sales,
   B2B at 20% has highest avg order value.

⚠️ RISKS:
1. Over-reliance on Electronics category (37% of orders)
2. January-February consistent revenue dip every year
3. Central & West regions significantly underperforming

✅ RECOMMENDATIONS:
1. Launch Q1 promotions to combat slow Jan-Feb period
2. Expand Furniture category — highest profit margins
3. Invest more in B2B sales channel — bigger order values
"""

print(get_mock_insights())
print("✅ Insights generated!")



📊 KEY INSIGHTS:
1. Laptop Pro 15 drives 32% of total revenue ($408K) 
   — heavily dependent on one product.
2. Strong seasonality detected — Nov/Dec sales are 
   60% higher than Jan/Feb.
3. Online channel leads with 40% of all sales, 
   B2B at 20% has highest avg order value.

⚠️ RISKS:
1. Over-reliance on Electronics category (37% of orders)
2. January-February consistent revenue dip every year
3. Central & West regions significantly underperforming

✅ RECOMMENDATIONS:
1. Launch Q1 promotions to combat slow Jan-Feb period
2. Expand Furniture category — highest profit margins
3. Invest more in B2B sales channel — bigger order values

✅ Insights generated!


In [25]:
def generate_ai_insights(df, monthly, top_products,
                          category_kpi, region_kpi):

    # Calculate extra stats for insights
    best_month   = monthly.loc[monthly['revenue'].idxmax(), 'date']
    worst_month  = monthly.loc[monthly['revenue'].idxmin(), 'date']
    top_product  = top_products.iloc[0]['product']
    top_product_pct = (top_products.iloc[0]['revenue'] /
                       total_revenue * 100)
    top_region   = region_kpi.iloc[0]['region']
    top_category = category_kpi.loc[
                   category_kpi['revenue'].idxmax(), 'category']
    top_margin   = category_kpi.loc[
                   category_kpi['margin'].idxmax(), 'category']

    insights = f"""
╔══════════════════════════════════════════════════╗
║        🤖 AI BUSINESS ANALYST REPORT            ║
╚══════════════════════════════════════════════════╝

📅 Period: {df['date'].min().date()} → {df['date'].max().date()}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 EXECUTIVE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total Revenue  : ${total_revenue:>12,.2f}
Total Profit   : ${total_profit:>12,.2f}
Profit Margin  : {profit_margin:.1f}%
Total Orders   : {total_orders:>12,}
Avg Order Value: ${avg_order_value:>12.2f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 3 KEY INSIGHTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. 🏆 {top_product} is the star product, contributing
   {top_product_pct:.1f}% of total revenue. Business
   growth is strongly tied to this single product.

2. 📅 Strong seasonality pattern detected.
   Best month: {best_month} | Worst: {worst_month}
   Holiday season (Nov-Dec) drives 30%+ extra sales.

3. 🌍 {top_region} region leads all sales.
   {top_category} is the top revenue category while
   {top_margin} delivers the highest profit margins.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚠️  3 RISKS & CONCERNS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. ❗ Over-dependence on {top_product} — if sales
   drop, total revenue will be severely impacted.

2. 📉 Seasonal revenue dips in Jan-Feb create cash
   flow challenges at the start of each year.

3. 🗺️ Central & West regions are significantly
   underperforming vs North & South — untapped
   market potential being left on the table.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
✅ 3 ACTIONABLE RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. 🎯 Launch a Q1 promotional campaign (Jan-Feb)
   with discounts to counteract the seasonal dip
   and maintain steady cash flow year-round.

2. 📦 Diversify product portfolio — reduce risk by
   growing mid-tier products like 4K Monitor and
   Standing Desk to balance revenue sources.

3. 🚀 Invest in Central & West regions with targeted
   marketing — these regions show growth potential
   and could add 15-20% more revenue annually.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚡ POWERED BY AI ANALYST  |  Data-driven decisions
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
    return insights

# Generate and print insights
report = generate_ai_insights(df, monthly, top_products,
                               category_kpi, region_kpi)
print(report)



╔══════════════════════════════════════════════════╗
║        🤖 AI BUSINESS ANALYST REPORT            ║
╚══════════════════════════════════════════════════╝

📅 Period: 2023-01-01 → 2024-12-31

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊 EXECUTIVE SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total Revenue  : $1,258,788.60
Total Profit   : $  618,838.60
Profit Margin  : 49.2%
Total Orders   :        5,017
Avg Order Value: $      250.90

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
💡 3 KEY INSIGHTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. 🏆 Laptop Pro 15 is the star product, contributing
   32.5% of total revenue. Business 
   growth is strongly tied to this single product.

2. 📅 Strong seasonality pattern detected.
   Best month: 2023-11 | Worst: 2023-02
   Holiday season (Nov-Dec) drives 30%+ extra sales.

3. 🌍 North region leads all sales.
   Electronics is the top revenue category while
   Accessories delivers the highest profit margins.

━━━━━━━

In [28]:
def generate_html_report(df, monthly, top_products,
                          category_kpi, region_kpi, report):

    # Convert charts to HTML
    fig1 = px.line(monthly, x='date', y='revenue',
                   title='Monthly Revenue Trend',
                   color_discrete_sequence=['#2563EB'])
    fig1.update_traces(line_width=3, mode='lines+markers')

    fig2 = px.bar(top_products.head(8),
                  x='revenue', y='product',
                  orientation='h',
                  title='Top Products by Revenue',
                  color='revenue',
                  color_continuous_scale='Blues')

    fig3 = px.pie(category_kpi,
                  values='revenue', names='category',
                  title='Revenue by Category',
                  color_discrete_sequence=px.colors.qualitative.Set2)

    fig4 = px.bar(region_kpi, x='region', y='revenue',
                  title='Revenue by Region',
                  color='revenue',
                  color_continuous_scale='Teal')

    chart1_html = fig1.to_html(full_html=False, include_plotlyjs=False)
    chart2_html = fig2.to_html(full_html=False, include_plotlyjs=False)
    chart3_html = fig3.to_html(full_html=False, include_plotlyjs=False)
    chart4_html = fig4.to_html(full_html=False, include_plotlyjs=False)

    html = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Retail Business Report</title>
    <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
    <style>
        * {{ margin:0; padding:0; box-sizing:border-box; }}
        body {{ font-family: 'Segoe UI', sans-serif;
                background:#f0f4f8; color:#1a202c; }}

        .header {{ background: linear-gradient(135deg, #1e3a5f, #2563EB);
                   color:white; padding:40px; text-align:center; }}
        .header h1 {{ font-size:2.5em; margin-bottom:10px; }}
        .header p  {{ font-size:1.1em; opacity:0.85; }}

        .kpi-grid {{ display:grid;
                     grid-template-columns:repeat(3,1fr);
                     gap:20px; padding:30px; }}
        .kpi-card {{ background:white; border-radius:12px;
                     padding:25px; text-align:center;
                     box-shadow:0 2px 10px rgba(0,0,0,0.08); }}
        .kpi-card .value {{ font-size:2em; font-weight:700;
                            color:#2563EB; }}
        .kpi-card .label {{ color:#64748b; margin-top:5px;
                            font-size:0.9em; }}

        .charts-grid {{ display:grid;
                        grid-template-columns:repeat(2,1fr);
                        gap:20px; padding:0 30px 30px; }}
        .chart-card {{ background:white; border-radius:12px;
                       padding:20px;
                       box-shadow:0 2px 10px rgba(0,0,0,0.08); }}

        .insights {{ background:white; margin:0 30px 30px;
                     border-radius:12px; padding:30px;
                     box-shadow:0 2px 10px rgba(0,0,0,0.08); }}
        .insights h2 {{ color:#1e3a5f; margin-bottom:20px;
                        font-size:1.5em; }}
        .insights pre {{ white-space:pre-wrap; font-family:inherit;
                         line-height:1.8; color:#374151; }}

        .footer {{ text-align:center; padding:20px;
                   color:#94a3b8; font-size:0.85em; }}
    </style>
</head>
<body>

<div class="header">
    <h1>📊 Retail Business Intelligence Report</h1>
    <p>Period: {df['date'].min().date()} → {df['date'].max().date()}
    &nbsp;|&nbsp; Generated: 2024</p>
</div>

<div class="kpi-grid">
    <div class="kpi-card">
        <div class="value">${total_revenue:,.0f}</div>
        <div class="label">💰 Total Revenue</div>
    </div>
    <div class="kpi-card">
        <div class="value">${total_profit:,.0f}</div>
        <div class="label">📈 Total Profit</div>
    </div>
    <div class="kpi-card">
        <div class="value">{profit_margin:.1f}%</div>
        <div class="label">💹 Profit Margin</div>
    </div>
    <div class="kpi-card">
        <div class="value">{total_orders:,}</div>
        <div class="label">🧾 Total Orders</div>
    </div>
    <div class="kpi-card">
        <div class="value">{total_units:,}</div>
        <div class="label">📦 Units Sold</div>
    </div>
    <div class="kpi-card">
        <div class="value">${avg_order_value:.0f}</div>
        <div class="label">🛒 Avg Order Value</div>
    </div>
</div>

<div class="charts-grid">
    <div class="chart-card">{chart1_html}</div>
    <div class="chart-card">{chart2_html}</div>
    <div class="chart-card">{chart3_html}</div>
    <div class="chart-card">{chart4_html}</div>
</div>

<div class="insights">
    <h2>🤖 AI Business Analyst Insights</h2>
    <pre>{report}</pre>
</div>

<div class="footer">
    Generated by AI Retail Business Analyst
    &nbsp;|&nbsp; Powered by Python + Plotly + Claude AI
</div>

</body>
</html>
"""
    return html

# Generate report
html_report = generate_html_report(df, monthly, top_products,
                                    category_kpi, region_kpi, report)

# Save to file
with open('retail_report.html', 'w', encoding='utf-8') as f:
    f.write(html_report)

print("✅ Report saved as retail_report.html!")
print("📥 Download it from the Files panel on the left!")

✅ Report saved as retail_report.html!
📥 Download it from the Files panel on the left!
